In [ ]:
# Colab setup -- installs SoftMobility when running on Google Colab.
# Safe to run locally: it does nothing outside Colab.
try:
    import google.colab  # noqa: F401

    %pip install -q git+https://github.com/C0PEP0D/SoftMobility.git
except ImportError:
    pass

# Tutorial 02. Mobility matrix of a rigid body

Testing the `SoftBody` class with no degrees-of-freedom and no flow.

In Stokes regime, the **mobility matrix** $\mathbf{M}$ of a rigid body relates non-hydrodynamic forces and torques $(\mathbf{F}, \mathbf{T})$ to the  translational and rotational velocities $(\mathbf{U}, \boldsymbol{\Omega})$ :

$$\begin{pmatrix}\mathbf{U}\\ \boldsymbol{\Omega}\end{pmatrix} = \mathbf{M} \begin{pmatrix}\mathbf{F}\\ \mathbf{T}\end{pmatrix}$$

For an assembly of $N$ beads, the library builds a $6N \times 6N$ **grand mobility matrix** using the Rotne–Prager–Yamakawa (RPY) approximation, which captures pairwise hydrodynamic interactions between beads.
This tutorial shows how to reduce it to a single $6 \times 6$ effective mobility for a **rigid** assembly (no internal degrees of freedom), and how to locate the special reference points called **hydrodynamic centers**.

In [ ]:
import numpy as np

import softmobility as sm

## 1. Input with YAML format

The assembly consists of three spheres of different radii arranged asymmetrically.
Because no `dof_names` are declared, the body is **rigid**: it has no internal degrees of freedom.

The `xref` design parameters define the **reference point** of the assembly.
Since all sphere positions are expressed in terms of `xref`, varying these parameters rigidly translates the entire assembly, effectively shifting which point in space serves as the body's frame reference point $\mathbf{r}_0$.

In [ ]:
yaml_data = """
design_names: [xref]

defaults:
    xref0: 0.0
    xref1: 0.0
    xref2: 0.0

spheres:
  - # Sphere 0
    radius: 1.0
    position: [xref0, xref1, xref2]

  - # Sphere 1
    radius: 0.75
    position: [xref0 +1.75, xref1, xref2]

  - # Sphere 2
    radius: 0.5
    position: [xref0, xref1 + 1.5, xref2]
"""

In [ ]:
# Creating sm.SoftBody object
mybody = sm.SoftBody(yaml_data)

## 2. Calculating the rigid mobility matrix

The method `compute_grand_mobility()` evaluates the $6N \times 6N$ grand mobility tensor using the RPY approximation.
For $N = 3$ spheres this yields an $18 \times 18$ matrix coupling all translational and rotational degrees of freedom of every bead.

In [ ]:
M_grand = mybody.compute_grand_mobility()
print("Grand mobility tensor M shape:", M_grand.shape)

The method `compute_rigid_tensors()` returns the rigid-body counterpart of the full mobility problem (all internal degrees of freedom frozen).
Its `Mred` field is the $6 \times 6$ effective mobility of the assembly: it maps translational and rotational velocities to force and torque applied at the assembly reference point $\mathbf{r}_0$ (of coordinate $[0,0,0]$ in the body's frame).

In [ ]:
M_rigid = np.array(mybody.compute_rigid_tensors().Mred)

print("Rigid-body mobility matrix (forces/torques at ref point r_O):")
print(M_rigid)

## 3. Assert the properties of the mobility matrix

The mobility matrix has two properties that we can check:

- **Symmetry**: a consequence of the Lorentz reciprocal theorem in Stokes flow: swapping the roles of force and velocity leaves the power dissipation invariant.
- **Positive definiteness**: ensures that viscous drag always dissipates energy: $\mathbf{F}^\top\cdot \mathbf{M}^{-1}\cdot \mathbf{F} \geq 0$ for all $\mathbf{F} \neq \mathbf{0}$.

In [ ]:
# Check if the matrix is symmetric
assert np.allclose(M_rigid, M_rigid.T), "M_mean is not symmetric"

# Check if the matrix is positive semi-definite (all eigenvalues >= 0)
eigvals = np.linalg.eigvalsh(M_rigid)
print("Eigenvalues of M:", eigvals)
assert np.all(eigvals >= 0), "M_mean is not positive definite"

print("The rigid mobility matrix is symmetric definite positive.")

## 4. Hydrodynamic centers

The $6 \times 6$ mobility and resistance matrices have the block structure

$$\mathbf{M} = \begin{pmatrix} \mathbf{a} & \mathbf{b}^\top \\ \mathbf{b} & \mathbf{c} \end{pmatrix}, \qquad \mathbf{R} \equiv \mathbf{M}^{-1} = \begin{pmatrix} \mathbf{A} & \mathbf{B}^\top \\ \mathbf{B} & \mathbf{C} \end{pmatrix},$$

where the $3\times 3$ off-diagonal blocks $\mathbf{b}$ and $\mathbf{B}$ couple translation to rotation.
Their asymmetry with respect to an arbitrary reference point reflects the choice of origin.

Choosing a special reference point for forces and torques makes the coupling block symmetric.
This special point is called the **hydrodynamic center**.
Two such centers exist (Kim & Karrila, Secs. 5.2.2 & 5.3.2):

- **Center of resistance** $\mathbf{x}_{CR}$: symmetrizes $\mathbf{B}$ in the resistance tensor. $$\mathbf{x}_{CR} = -\tfrac{1}{2}\bigl(\mathrm{tr}(\mathbf{A})\,\mathbf{I} - \mathbf{A}\bigr)^{-1} \cdot \boldsymbol{\epsilon} : (\mathbf{B} - \mathbf{B}^\top)$$

- **Center of mobility** $\mathbf{x}_{CM}$: symmetrizes $\mathbf{b}$ in the mobility tensor. $$\mathbf{x}_{CM} = \tfrac{1}{2}\bigl(\mathrm{tr}(\mathbf{c})\,\mathbf{I} - \mathbf{c}\bigr)^{-1} \cdot \boldsymbol{\epsilon} : (\mathbf{b} - \mathbf{b}^\top)$$

where $\boldsymbol{\epsilon}$ is the Levi-Civita tensor.

Both formulas give the position of the centre **relative to the current reference point**.
To make that centre the new reference, the body must therefore be translated by $-\mathbf{x}_{CR}$ (resp. $-\mathbf{x}_{CM}$).
To do so, we will change `xref` and check the symmetry of the resistance (resp. moblity) matrix.

In [ ]:
# Tensors to compute the mobility center
b = M_rigid[3:, :3]
c = M_rigid[3:, 3:]

# Tensors to compute the resistance center
resistance_matrix = np.linalg.inv(M_rigid)
A = resistance_matrix[:3, :3]
B = resistance_matrix[3:, :3]

levicivita = np.array(
    [
        [[0.0, 0.0, 0.0], [0.0, 0.0, 1.0], [0.0, -1.0, 0.0]],
        [[0.0, 0.0, -1.0], [0.0, 0.0, 0.0], [1.0, 0.0, 0.0]],
        [[0.0, 1.0, 0.0], [-1.0, 0.0, 0.0], [0.0, 0.0, 0.0]],
    ]
)

# From Kim & Karrila 5.2.2 and 5.3.2
trAItmA_inv = np.linalg.inv(np.trace(A) * np.eye(3) - A)
x_cr = -0.5 * np.einsum("ij,jkl,kl->i", trAItmA_inv, levicivita, B - B.transpose())
trcItmc_inv = np.linalg.inv(np.trace(c) * np.eye(3) - c)
x_cm = 0.5 * np.einsum("ij,jkl,kl->i", trcItmc_inv, levicivita, b - b.transpose())

print("Hydrodynamic center of resistance:", x_cr)
print("Hydrodynamic center of mobility:", x_cm)

We shift the assembly so that its centre of resistance coincides with the reference point, by setting `xref` to $-\mathbf{x}_{CR}$.
Since all sphere positions are defined relative to `xref`, this rigidly translates the whole assembly.
The rotation–translation block of the **resistance** tensor should now be symmetric.

In [ ]:
# Assert the resistance tensor with x_cr the reference point
print("BEFORE change of coordinates:")
R = np.linalg.inv(M_rigid)
B = R[3:, :3]
print("Rotation-translation component of the resistance tensor:")
print(B, "\n")


# Shift the assembly so the centre of resistance coincides with the reference point
mybody.set_design_defaults(new_dict={"xref0": -x_cr[0], "xref1": -x_cr[1], "xref2": -x_cr[2]})

M_rigid = np.array(mybody.compute_rigid_tensors().Mred)
R = np.linalg.inv(M_rigid)

print("\nAFTER change of coordinates:")
B = R[3:, :3]
print("Rotation-translation component of the resistance tensor:")
print(B)

# Check if the matrix is symmetric
if np.allclose(B, B.T, atol=1e-5):
    print("R is now symmetric")
else:
    raise ValueError("R is not symmetric after change of coordinates")

We repeat the verification for the **mobility** tensor, shifting the assembly so the centre of mobility coincides with the reference point (`xref` set to $-\mathbf{x}_{CM}$).

In [ ]:
# Assert the mobility tensor with x_cm the reference point
print("BEFORE change of coordinates:")
b = M_rigid[3:, :3]
print("Rotation-translation component of the mobility tensor:")
print(b, "\n")

# Shift the assembly so the centre of mobility coincides with the reference point
mybody.set_design_defaults(new_dict={"xref0": -x_cm[0], "xref1": -x_cm[1], "xref2": -x_cm[2]})

M_rigid = np.array(mybody.compute_rigid_tensors().Mred)

print("\nAFTER change of coordinates:")
b = M_rigid[3:, :3]
print("Rotation-translation component of the mobility tensor:")
print(b)

# Check if the matrix is symmetric
if np.allclose(b, b.T, atol=1e-5):
    print("M is now symmetric")
else:
    raise ValueError("M is not symmetric after change of coordinates")

## References
S. Kim and S. J. Karrila, *Microhydrodynamics: principles and selected applications* (Butterworth-Heinemann, 2013).